In [ ]:
include("../src/model.jl")

using SpecialFunctions: erf, erfc
using LinearAlgebra: diag, norm
using Test

using Plots, LaTeXStrings
default(fmt = :png)

In [ ]:
"""
Solve the problem in equilibrium
"""

# Single impurity Anderson model
siam = NCA.SIAM(
  V₀=sqrt(0.2),
  ε₀f=-0.3,
  tilde=true,
  Nc₀=x->NCA.lorentzian(x, 0.0, 1.0)
  )

# Equilibrium NCA solver
nca = NCA.FreqNCA(
  model=siam,
  β=5e2,
  mesh=NCA.LinearMesh(-4.0,4.0,2^14-1)
    )

# Calculate Green functions
GᶜA, GᶠA, GᵇA = begin
  
  imΣ_t = NCA.solve!(nca; project=false)
  
  Ginv = NCA.slave_particle_inv_gf(nca, imΣ_t)
  Gᵇ = 1.0 ./ Ginv[1]
  Gᶠ = 1.0 ./ Ginv[2]
  
  # Trim just the imaginary part
  imGᵇ = view(reinterpret(Float64, Gᵇ), 2:2:length(Gᵇ))
  imGᵇ[imGᵇ .< nca.eps] .= 0.0
  imGᶠ = view(reinterpret(Float64, Gᶠ), 2:2:length(Gᶠ))
  imGᶠ[imGᶠ .< nca.eps] .= 0.0
  
  Gᶜ = NCA.kramerskronig(nca.mesh, π * nca.data.Nc, :real) + 1im * π * nca.data.Nc;
  
  Gᶜ, Gᶠ, Gᵇ
end;

# plot(nca.mesh.mesh, GᵇA |> imag, label=L"$A_b$", title="Spectral functions", xlab=L"$\omega$")
# plot!(nca.mesh.mesh, GᶠA |> imag, label=L"$A_f$")
# plot!(nca.mesh.mesh, GᶜA |> imag, label=L"$A_c$")
# xlims!(-0.1,0.1)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
model = PhotonAssitedModel(
  #= SIAM parameters =#
  V₀=siam.V₀,
  ε₀f=siam.ε₀f,
  λ₀=siam.λ₀,
  σ=1.0,
  
  #= Light-field parameters =#
  Γ₀=0.00,  # light-matter coupling
  ω₀=1e-1, # photon-ground state energy
  δ₀=1e-1, # photon artificial width
  
  #= Photon-pulse parameters =#
  tc=15,   # peak arrival time
  Ω=1e0,  # frequency bandwitdh of the pulse
  
  #= SLH photon formulation =#
  Γ=5.     # pump and loss strength
)

# Pulse and SLH shape
model.ξ = t -> (model.Ω^2 / (2π))^(1/4)* exp(-(t - model.tc)^2 * (model.Ω^2 / 4))
model.γ = t -> model.Γ * model.ξ(t)
model.λ = t -> model.Γ * model.ξ(t) / sqrt(0.5 * erfc((t - model.tc) * model.Ω / sqrt(2)))
model.hᵧ= t -> (model.ω₀ - 0.5im * (model.λ(t) + model.γ(t)))

# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
data = let n₀ = 0.0 # initial photon-field occupation
  
  # Create a new frequency grid that supports for complex convolutions
  freqs = NCA.LinearMesh(nca.mesh.xmin, nca.mesh.xmax, nca.mesh.L, true);
  
  bose = NCA.nb.(freqs, nca.β)
  fermi = NCA.nf.(freqs, nca.β);
  boltz = NCA.bf.(freqs, nca.β) .|> Float64 ; boltz[boltz .== Inf] .= 0.0;
  
  # Integral over frequencies
  ∫dω(f) = integrate(freqs.mesh, f, TrapezoidalEvenFast()) / (2π)
  
  GᶜL = ∫dω(fermi .* (GᶜA - conj(GᶜA)))
  GᶜL = GreenFunction(GᶜL * ones(1,1), Lesser)
  
  GᶜG = -(1.0im - ∫dω(fermi .* (GᶜA - conj(GᶜA))))
  GᶜG = GreenFunction(GᶜG * ones(1,1), Greater)
  
  GᶠL = ∫dω(boltz .* (GᶠA - conj(GᶠA)))
  GᶠL = GreenFunction(GᶠL * ones(1,1), Lesser)

  GᶠG = -(1.0im)#- ζ∫dω(fermi .* (GᶠA - conj(GᶠA))))
  GᶠG = GreenFunction(GᶠG * ones(1,1), Greater)

  GᵇL = -∫dω(boltz .* (GᵇA - conj(GᵇA)))
  GᵇL = GreenFunction(GᵇL * ones(1,1), Lesser)
  
  GᵇG = -(1.0im)# + ζ∫dω(bose .* (GᵇA - conj(GᵇA))))
  GᵇG = GreenFunction(GᵇG * ones(1,1), Greater)
  
  GᵞL = -1.0im * n₀
  GᵞL = GreenFunction(GᵞL * ones(1,1), Lesser)
  
  GᵞG = -1.0im * (1 + n₀)
  GᵞG = GreenFunction(GᵞG * ones(1,1), Greater)
  
  Gᶜ˥m = GreenFunction(ElasticArray(hcat(GᶜA)), MixedLesser)
  Gᶜ˥p = GreenFunction(ElasticArray(hcat(conj(GᶜA))), MixedLesser)
  
  Gᶠ˥m = GreenFunction(ElasticArray(hcat(GᶠA)), MixedLesser)
  Gᶠ˥p = GreenFunction(ElasticArray(hcat(conj(GᶠA))), MixedLesser)
  
  Gᵇ˥m = GreenFunction(ElasticArray(hcat(GᵇA)), MixedLesser)
  Gᵇ˥p = GreenFunction(ElasticArray(hcat(conj(GᵇA))), MixedLesser)
  
  # Spectral report
  println("Equilibrium spectral function normalizations:")
  println(" ∫dω Aᶜ(ω) = ", ∫dω(GᶜA - conj(GᶜA)) |> imag, ", nᶜ = ", GᶜL[1,1] |> imag)
  println(" ∫dω Aᶠ(ω) = ", ∫dω(GᶠA - conj(GᶠA)) |> imag, ", nᶠ = ", GᶠL[1,1] |> imag)
  println(" ∫dω Aᵇ(ω) = ", ∫dω(GᵇA - conj(GᵇA)) |> imag, ", nᵇ = ", -GᵇL[1,1] |> imag)
  
  ProblemData(
    GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,
    Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,
    freqs,fermi,bose,boltz);
  end;

In [ ]:
# Initial condition
u0 = tuple(
  [data.GᶜL,data.GᶜG,data.GᶠL,data.GᶠG,data.GᵇL,data.GᵇG,data.GᵞL,data.GᵞG],
  [data.Gᶜ˥m,data.Gᶜ˥p,data.Gᶠ˥m,data.Gᶠ˥p,data.Gᵇ˥m,data.Gᵇ˥p]
)

# Vertical step
f_vert(u, times, t1::Int, t2::Int) = f_vert_time(model, data, times, t1, t2)
f_vert(u, v, times, t1::Int, t2::Int) = f_vert(u, times, t1, t2) .+ f_vert_freq(model, data, times, t1, t2)

# Diagonal step
f_diag(u, times, t1::Int) = (println("t: $(times[t1])"); f_diag_time(model, data, times, t1))
f_diag(u, v, times, t1::Int) = f_diag(u, times, t1) .+ update_norms!(f_diag_freq(model, data, times, t1))

# Vertical step for mixed functions
f_line(u, v, times, t1::Int) = f_line_freq(model, data, times, t1)

thermal_norms = float(Inf)
update_norms!(n) = (global thermal_norms; thermal_norms = norm.(n); n)

"""
Step until the thermal contribution is small and then discard it
"""

# Criteria for evaluating the thermal collision integrals
# TODO: Accumulate norms over horizontal step (don't take into account just the diagonal)
thermal_criteria(norms) = max(norms...) < 1e-3

tspan = (0.0, 30.0)

# The solver will stop when tspan[2] is reached or `stop` evaluates to `false`
state, caches = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; f_line=f_line,
    init_dt=1e-3, atol=1e-6, rtol=1e-4,
    stop=()->thermal_criteria(thermal_norms),
    w_line! =(t1)->calculate_Σ_freq!(model,data,t1),
    w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

@testset "KMS boundary conditions" begin
  @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,fermi,bose,boltz,freqs = data
  
  # G^R(t,t) = -i
  @test all(diag(GᶜG.data - GᶜL.data) .≈ -1.0im)
  @test all(diag(GᵞG.data - GᵞL.data) .≈ -1.0im)
  @test all(diag(GᶠG.data) .≈ -1.0im)
  @test all(diag(GᵇG.data) .≈ -1.0im)

  ∫dω(g, f) = (f[abs.(imag(f)) .< 1e-12] .= 0.0; integrate(freqs.mesh, g .* f, TrapezoidalEvenFast()) / (2π))
  ≅(a,b) = isapprox(a,b;atol=1e-2)
  
  # C^<(t,0) = C˥(t,0+)
  @test all(GᶜL[t,1] ≅ ∫dω(fermi, Gᶜ˥m[t] - Gᶜ˥p[t]) for t in 1:size(Gᶜ˥m,2))
  @test all(GᶠL[t,1] ≅ ∫dω(boltz, Gᶠ˥m[t] - Gᶠ˥p[t]) for t in 1:size(Gᶠ˥m,2))
  @test all(GᵇL[t,1] ≅ ∫dω(-boltz, Gᵇ˥m[t] - Gᵇ˥p[t]) for t in 1:size(Gᵇ˥m,2))
end

# If we don't need to evaluate the thermal collision integrals anymore
if thermal_criteria(thermal_norms)
  
  # We need a new integrator-state without the mixed functions
  state = KadanoffBaym.VCABMState((state.u[1],), state.t, state.dt)
  
  state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
    atol=1e-6, rtol=1e-4,
    w_time! = (t1,t2) -> calculate_Σ_time!(model,data,t1,t2),
    st_ch = (state, caches))
end

# state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
#   atol=1e-6, rtol=1e-4, 
#   w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

# Extract info
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,freqs = data;
times = state.t;

In [ ]:
# Pulse
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, model.ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
# Mixed spectral functions
plot(nca.mesh.mesh, Gᶜ˥m.data[:,1] - Gᶜ˥p.data[:,1] |> imag)
plot!(nca.mesh.mesh, Gᶜ˥m.data[:,end] - Gᶜ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᵇ˥m.data[:,1] - Gᵇ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᵇ˥m.data[:,end] - Gᵇ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᶠ˥m.data[:,1] - Gᶠ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᶠ˥m.data[:,end] - Gᶠ˥p.data[:,end] |> imag .|> -)

In [ ]:
# Slave-particle occupation
plot(times,-imag(diag(GᵇL.data)))
plot!(twinx(),times,imag(diag(GᶠL.data)), color=:orange)

In [ ]:
# Time-dependent spectral functions
ρ = wigner_transform(GᵞG.data - GᵞL.data)
wigner_plot!(real(ρ), times)

# ρ = wigner_transform(GᵇG.data - GᵇL.data)#; Nf=4096*2);
# wigner_plot!(real(ρ), times)#; ylims=(-10,10))

# ρ = wigner_transform(GᶠG.data - GᶠL.data)#; Nf=4096*2);
# wigner_plot!(real(ρ), times)#; ylims=(-10,10))